# Fake News Detection Using Machine Learning
**Author:** Sairam Odela  
**Algorithm:** Logistic Regression with TF-IDF  
**Dataset:** Kaggle Fake and Real News Dataset  

This notebook covers EDA, text preprocessing, model training, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded!')

## 1. Load Dataset

In [ ]:
fake_df = pd.read_csv('../data/Fake.csv')
true_df = pd.read_csv('../data/True.csv')

fake_df['label'] = 0
true_df['label'] = 1

df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.shape)
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Class distribution
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#ef4444','#10b981'])
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Fake','Real'], rotation=0)

# Article length distribution
df['text_len'] = df['text'].str.split().str.len()
df.groupby('label')['text_len'].plot(kind='hist', ax=axes[1], bins=50, alpha=0.7)
axes[1].set_title('Article Length Distribution')
axes[1].legend(['Fake','Real'])

# Subject distribution
df['subject'].value_counts().plot(kind='bar', ax=axes[2], color='#0d6efd')
axes[2].set_title('News Subject Distribution')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"Avg fake article length : {df[df.label==0]['text_len'].mean():.0f} words")
print(f"Avg real article length : {df[df.label==1]['text_len'].mean():.0f} words")

In [ ]:
# Word Clouds
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, label, title, color in zip(axes, [0, 1], ['Fake News', 'Real News'], ['Reds', 'Greens']):
    text = ' '.join(df[df['label']==label]['text'].dropna().values)
    wc = WordCloud(width=800, height=400, background_color='white',
                   colormap=color, max_words=100).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Text Preprocessing & Feature Engineering

In [ ]:
import sys
sys.path.insert(0, '../src')
from preprocess import clean_text

df['content'] = (df['title'].fillna('') + ' ' + df['text'].fillna('')).apply(clean_text)
print('Sample cleaned text:')
print(df['content'].iloc[0][:300])

## 4. Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['content'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2),
                              stop_words='english', sublinear_tf=True, min_df=2)),
    ('clf',   LogisticRegression(max_iter=1000, C=5.0, solver='lbfgs', random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print(classification_report(y_test, y_pred, target_names=['Fake','Real']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Fake','Real']).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix')

# Top TF-IDF features
tfidf     = pipeline.named_steps['tfidf']
clf       = pipeline.named_steps['clf']
features  = tfidf.get_feature_names_out()
coefs     = clf.coef_[0]
top_real  = np.argsort(coefs)[-15:][::-1]
top_fake  = np.argsort(coefs)[:15]

axes[1].barh([features[i] for i in top_real], coefs[top_real], color='#10b981', label='Real')
axes[1].barh([features[i] for i in top_fake], coefs[top_fake], color='#ef4444', label='Fake')
axes[1].set_title('Top Discriminative Words')
axes[1].legend()

plt.tight_layout()
plt.show()

# Cross-validation
cv = cross_val_score(pipeline, df['content'], df['label'], cv=5, scoring='accuracy')
print(f'5-Fold CV Accuracy: {cv.mean()*100:.2f}% ± {cv.std()*100:.2f}%')